# Aula 4 — NLP: Vetorização de Texto e Word Embeddings

**CCM-109 · Tópicos Avançados em Inteligência Artificial · UFABC**  
Adaptado do material MIT 15.773 Hands-on Deep Learning (Farias & Ramakrishnan, 2024)

---

Este notebook cobre:

1. **Setup e dataset** — letras de música (hip-hop / pop / rock)
2. **Pipeline STIE** — Standardize → Tokenize → Index → Encode
3. **Bag of Words com Keras** — `TextVectorization`, multi-hot, bigramas, dropout
4. **Word Embeddings pré-treinados** — GloVe (Stanford), similaridade e analogias
5. **Propriedades geométricas** — gênero, capital, superlativo, tempo verbal
6. **Visualização t-SNE** — famílias de palavras em 2D
7. **Classificação com Keras** — GloVe frozen vs fine-tuned vs embedding do zero
8. **Classificação com PyTorch** — `nn.Embedding` + GAP, comparação final

---
## 0. Setup

In [ ]:
!pip install -q gensim scikit-learn matplotlib pandas

In [ ]:
import re, string, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from collections import Counter

import tensorflow as tf
from tensorflow import keras

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from sklearn.manifold import TSNE
from sklearn.metrics import classification_report

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'TensorFlow {tf.__version__} | PyTorch {torch.__version__} | device={device}')

---
## 1. Dataset — Letras de Música

Usaremos um dataset com ~90 k letras de música classificadas em **hip-hop**, **pop** e **rock**.

In [ ]:
train_url = "https://www.dropbox.com/scl/fi/ito6bnl2yaf1uw0uqibzf/lyric_genre_train.csv?rlkey=04dkn5un2djza8x0bdmfnlw3u&st=y47qh8i4&dl=1"
val_url   = "https://www.dropbox.com/scl/fi/xmywjzqsaa8n5sn1bs0t9/lyric_genre_val.csv?rlkey=hggbeo0s1iaxjpa6z80429xl9&st=6i7d8eau&dl=1"
test_url  = "https://www.dropbox.com/scl/fi/fnocl69w9ojs9s5zb0xvf/lyric_genre_test.csv?rlkey=z4hjopw7vaihoh948cbb5mvdp&st=xwond7dp&dl=1"

train_df = pd.read_csv(train_url, index_col=0)
val_df   = pd.read_csv(val_url,   index_col=0)
test_df  = pd.read_csv(test_url,  index_col=0)

print(f'Train: {train_df.shape[0]:,} | Val: {val_df.shape[0]:,} | Test: {test_df.shape[0]:,}')
print(f'\nColunas: {list(train_df.columns)}')
train_df.head()

In [ ]:
# Distribuição das classes
dist = train_df['Genre'].value_counts()
print('Distribuição no treino:')
print(dist)
print(f'\n→ Baseline majoritário (Rock): {dist.max()/dist.sum():.1%}')

# Exemplo de cada classe
for genre in ['Hip Hop', 'Pop', 'Rock']:
    sample = train_df[train_df['Genre'] == genre]['Lyric'].iloc[0]
    print(f'\n--- {genre} ---')
    print(sample[:200])

In [ ]:
# Codifica rótulos (one-hot para Keras, inteiro para PyTorch)
y_train_oh = pd.get_dummies(train_df['Genre']).to_numpy().astype('float32')
y_val_oh   = pd.get_dummies(val_df['Genre']).to_numpy().astype('float32')
y_test_oh  = pd.get_dummies(test_df['Genre']).to_numpy().astype('float32')

label_names = list(pd.get_dummies(train_df['Genre']).columns)  # ['Hip Hop', 'Pop', 'Rock']
label2idx   = {l: i for i, l in enumerate(label_names)}

y_train_int = train_df['Genre'].map(label2idx).to_numpy()
y_val_int   = val_df['Genre'].map(label2idx).to_numpy()
y_test_int  = test_df['Genre'].map(label2idx).to_numpy()

print('Classes:', label_names)
print('Shape y_train_oh:', y_train_oh.shape)

---
## 2. Pipeline STIE — Standardize → Tokenize → Index → Encode

O pipeline padrão de NLP transforma texto bruto em vetores numéricos em quatro etapas.

| Etapa | O que faz |
|---|---|
| **Standardize** | Minúsculas, remove pontuação, stop words |
| **Tokenize** | Divide em tokens (palavras, subpalavras, n-gramas) |
| **Index** | Mapeia cada token a um inteiro único |
| **Encode** | Mapeia inteiro → vetor (one-hot, embedding…) |

In [ ]:
# ── Demonstração manual do STIE ───────────────────────────────────────────────
example = "Hola! What do you picture when you think of traveling to Mexico?"

# S — Standardize
std = example.lower()
std = re.sub(r'[^\w\s]', '', std)
print(f'S: {std}')

# T — Tokenize
tokens = std.split()
print(f'T: {tokens}')

# I — Index (vocabulário mínimo)
mini_vocab = {w: i for i, w in enumerate(['<PAD>', '<UNK>'] + sorted(set(tokens)))}
indexed = [mini_vocab.get(t, 1) for t in tokens]
print(f'I: {indexed}')

# E — Encode (one-hot)
V = len(mini_vocab)
onehot = np.eye(V)[indexed]
print(f'E (one-hot): matriz {onehot.shape} — {V} dimensões, {np.count_nonzero(onehot)/onehot.size:.1%} denso')

In [ ]:
# ── STIE com Keras TextVectorization ─────────────────────────────────────────
tv_demo = keras.layers.TextVectorization(output_mode='multi_hot', standardize='lower_and_strip_punctuation')

haiku = [
    "I write, erase, rewrite",
    "Erase again, and then",
    "A poppy blooms.",
]
tv_demo.adapt(haiku)

print('Vocabulário:', tv_demo.get_vocabulary())
print('Tamanho:', len(tv_demo.get_vocabulary()))

test_sent = "I write, rewrite, and still rewrite again"
encoded = tv_demo([test_sent])
print(f'\nFrase: "{test_sent}"')
print(f'Multi-hot: {encoded.numpy()[0]}')
print('(índice 0 = [UNK], demais = palavras do vocabulário)')

---
## 3. Bag of Words com Keras

### 3.1 Multi-hot (unigramas)

In [ ]:
MAX_TOKENS_BOW = 5000

text_vec_bow = keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS_BOW,
    output_mode='multi_hot'
)
text_vec_bow.adapt(train_df['Lyric'])

print(f'Vocabulário: {len(text_vec_bow.get_vocabulary()):,} tokens')
print('20 mais comuns:', text_vec_bow.get_vocabulary()[:20])
print('20 menos comuns:', text_vec_bow.get_vocabulary()[-20:])

X_train_bow = text_vec_bow(train_df['Lyric'])
X_val_bow   = text_vec_bow(val_df['Lyric'])
X_test_bow  = text_vec_bow(test_df['Lyric'])

print(f'\nShape X_train: {X_train_bow.shape}  (cada letra → vetor de {MAX_TOKENS_BOW}d)')

In [ ]:
# Modelo BoW simples — Dense 8 → Dense 3
inputs  = keras.Input(shape=(MAX_TOKENS_BOW,))
x       = keras.layers.Dense(8, activation='relu')(inputs)
outputs = keras.layers.Dense(3, activation='softmax')(x)

model_bow = keras.Model(inputs, outputs)
model_bow.summary()

model_bow.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

hist_bow = model_bow.fit(
    X_train_bow, y_train_oh,
    validation_data=(X_val_bow, y_val_oh),
    epochs=5, batch_size=32, verbose=1
)

loss_bow, acc_bow = model_bow.evaluate(X_test_bow, y_test_oh, verbose=0)
print(f'\nBoW Test accuracy: {acc_bow:.4f}')

### 3.2 Bigramas + Dropout

Bigramas capturam contexto local: "not good" é diferente de "not" + "good" separados.

> **Dropout** desativa aleatoriamente uma fração dos neurônios durante o treino, forçando redundância e reduzindo overfitting.

In [ ]:
# Demonstração de bigramas
tv_bigram_demo = keras.layers.TextVectorization(ngrams=2, max_tokens=20, output_mode='multi_hot')
tv_bigram_demo.adapt(['the cat sat on the mat.'])
print('Vocabulário com bigramas:', tv_bigram_demo.get_vocabulary())

In [ ]:
MAX_TOKENS_BIGRAM = 20000

text_vec_bigram = keras.layers.TextVectorization(
    ngrams=2,
    max_tokens=MAX_TOKENS_BIGRAM,
    output_mode='multi_hot'
)
text_vec_bigram.adapt(train_df['Lyric'])

# tf.data — vetoriza em batches para evitar matriz 49k×20k (~3.9 GB) na memória
def make_dataset_bi(df, labels, batch_size=32, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((df['Lyric'].values, labels))
    if shuffle:
        ds = ds.shuffle(len(df), seed=42)
    ds = ds.batch(batch_size)
    ds = ds.map(lambda x, y: (text_vec_bigram(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds_bi = make_dataset_bi(train_df, y_train_oh, shuffle=True)
val_ds_bi   = make_dataset_bi(val_df,   y_val_oh)
test_ds_bi  = make_dataset_bi(test_df,  y_test_oh)

print(f'Vocabulário bigrama: {len(text_vec_bigram.get_vocabulary()):,} tokens')

# Modelo com Dropout
inputs  = keras.Input(shape=(MAX_TOKENS_BIGRAM,))
x       = keras.layers.Dense(8, activation='relu')(inputs)
x       = keras.layers.Dropout(0.5)(x)
outputs = keras.layers.Dense(3, activation='softmax')(x)

model_bigram = keras.Model(inputs, outputs)
model_bigram.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

hist_bigram = model_bigram.fit(
    train_ds_bi,
    validation_data=val_ds_bi,
    epochs=5, verbose=1
)

loss_bi, acc_bi = model_bigram.evaluate(test_ds_bi, verbose=0)
print(f'
Bigrama+Dropout Test accuracy: {acc_bi:.4f}')

In [ ]:
# ── Previsão interativa ───────────────────────────────────────────────────────
def lyric_predict(phrase, model=model_bigram, vectorizer=text_vec_bigram):
    vect = vectorizer([phrase])
    probs = model.predict(vect, verbose=0)[0]
    print(f'Hip-Hop: {probs[0]*100:.1f}%')
    print(f'Pop:     {probs[1]*100:.1f}%')
    print(f'Rock:    {probs[2]*100:.1f}%')
    print(f'→ Predição: {label_names[np.argmax(probs)]}')

# ABBA — Dancing Queen
lyric_predict("""
You can dance, you can jive, having the time of your life
See that girl, watch that scene, diggin' the dancing queen
Friday night and the lights are low,
Looking out for the place to go
""")

print()

# Hip-Hop
lyric_predict("""
I grew up on the crime side, the New York Times side
Staying alive was no jive, had secondhand, moms bounced on old man
Got our own thing, copped our own rings
""")

---
## 4. Word Embeddings Pré-treinados — GloVe

GloVe (Pennington et al., 2014) aprende vetores densos minimizando:

$$\mathcal{L} = \sum_{i,j} f(X_{ij})\left(\mathbf{w}_i^\top \mathbf{w}_j + b_i + b_j - \log X_{ij}\right)^2$$

Usaremos o modelo `glove.6B.100d` — 100 dimensões, treinado em Wikipedia + Gigaword (6 bilhões de tokens).

In [ ]:
# Baixa e descomprime o GloVe (~862 MB zip, ~340 MB arquivo 100d)
if not os.path.exists('glove.6B.100d.txt'):
    !wget -q http://nlp.stanford.edu/data/glove.6B.zip
    !unzip -q glove.6B.zip glove.6B.100d.txt
    print('GloVe baixado.')
else:
    print('GloVe já disponível.')

In [ ]:
# Carrega o arquivo GloVe em um dicionário  palavra → vetor numpy
EMBED_DIM = 100
embeddings_index = {}

with open('glove.6B.100d.txt', encoding='utf-8') as f:
    for line in f:
        word, coefs = line.split(maxsplit=1)
        embeddings_index[word] = np.fromstring(coefs, dtype='f', sep=' ')

print(f'Vetores carregados: {len(embeddings_index):,}')
print(f'Dimensão: {embeddings_index["movie"].shape}')
print(f'\nPrimeiros elementos de "movie": {embeddings_index["movie"][:8]}')

In [ ]:
# ── Similaridade por cosseno ──────────────────────────────────────────────────
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def most_similar(word, n=5):
    if word not in embeddings_index:
        return []
    vec = embeddings_index[word]
    sims = {w: cosine_sim(vec, v) for w, v in embeddings_index.items() if w != word}
    return sorted(sims.items(), key=lambda x: -x[1])[:n]

pares = [
    ('movie',  'film'),   ('movie',  'banana'),
    ('king',   'queen'),  ('france', 'paris'),
    ('good',   'best'),   ('good',   'bad'),
]

print(f'{"Par":<30} Cosine')
print('-' * 40)
for w1, w2 in pares:
    v1 = embeddings_index.get(w1)
    v2 = embeddings_index.get(w2)
    if v1 is not None and v2 is not None:
        sim = cosine_sim(v1, v2)
        bar = '█' * int(sim * 18)
        print(f'  ({w1}, {w2}):{" "*(20-len(w1)-len(w2))} {sim:.3f}  {bar}')

---
## 5. Propriedades Geométricas — Analogias

A operação **rei − homem + mulher ≈ rainha** demonstra que relações semânticas se tornam **direções consistentes** no espaço vetorial.

In [ ]:
# Carrega gensim para analogias convenientes (usa o mesmo arquivo GloVe)
from gensim.models import KeyedVectors
import gensim.downloader as api

print('Carregando modelo gensim (glove-wiki-gigaword-100)...')
glove = api.load('glove-wiki-gigaword-100')

def analogy(pos1, neg1, pos2, topn=3):
    """pos1 - neg1 + pos2 ≈ ?"""
    return glove.most_similar(positive=[pos1, pos2], negative=[neg1], topn=topn)

grupos = [
    # Categoria,  A,        B,      C,       descricao
    ('Gênero',     'king',   'man',   'woman',   'king − man + woman'),
    ('Gênero',     'actor',  'man',   'woman',   'actor − man + woman'),
    ('Capital',    'paris',  'france','germany', 'paris − france + germany'),
    ('Capital',    'paris',  'france','japan',   'paris − france + japan'),
    ('Superlativo','best',   'good',  'bad',     'best − good + bad'),
    ('Superlativo','biggest','big',   'small',   'biggest − big + small'),
    ('Tempo verb.','ran',    'run',   'walk',    'ran − run + walk'),
    ('Tempo verb.','went',   'go',    'come',    'went − go + come'),
    ('Antônimo',   'cold',   'hot',   'fast',    'cold − hot + fast'),
]

print(f'\n{"Categoria":<14} {"Operação":<28} {"Top-1":<14} Score')
print('=' * 65)
for cat, pos1, neg1, pos2, desc in grupos:
    try:
        result = analogy(pos1, neg1, pos2, topn=1)
        word, score = result[0]
        print(f'{cat:<14} {desc:<28} {word:<14} {score:.3f}')
    except KeyError as e:
        print(f'{cat:<14} {desc:<28} [erro: {e}]')

---
## 6. Visualização com t-SNE

t-SNE projeta os vetores de 100d em 2D preservando vizinhança local. Famílias de palavras relacionadas devem aparecer agrupadas.

In [ ]:
families = {
    'Realeza':     ['king', 'queen', 'prince', 'princess', 'throne', 'crown'],
    'Países':      ['france', 'germany', 'japan', 'brazil', 'italy', 'spain'],
    'Capitais':    ['paris', 'berlin', 'tokyo', 'brasilia', 'rome', 'madrid'],
    'Computação':  ['computer', 'software', 'algorithm', 'data', 'network', 'code'],
    'Esportes':    ['soccer', 'football', 'tennis', 'basketball', 'athlete', 'stadium'],
    'Superlativos':['good', 'better', 'best', 'bad', 'worse', 'worst'],
}
colors = ['#f59e0b', '#a78bfa', '#38bdf8', '#34d399', '#fb7185', '#fb923c']

words, vecs, color_map, labels = [], [], [], []
for (label, color), group_words in zip(zip(families.keys(), colors), families.values()):
    for w in group_words:
        if w in glove:
            words.append(w); vecs.append(glove[w])
            color_map.append(color); labels.append(label)

coords = TSNE(n_components=2, perplexity=8, random_state=42, n_iter=2000).fit_transform(np.array(vecs))

fig, ax = plt.subplots(figsize=(12, 8))
ax.set_facecolor('#0f172a'); fig.patch.set_facecolor('#0f172a')

for i, (x, y) in enumerate(coords):
    ax.scatter(x, y, color=color_map[i], s=80, zorder=3)
    ax.annotate(words[i], (x, y), fontsize=9, color='white', fontweight='bold',
                xytext=(5, 4), textcoords='offset points',
                path_effects=[pe.withStroke(linewidth=2, foreground='black')])

from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=c, label=l) for l, c in zip(families.keys(), colors)],
          loc='upper left', framealpha=0.3, facecolor='#1e293b', labelcolor='white', fontsize=9)
ax.set_title('t-SNE de embeddings GloVe (100d → 2d)', color='white', fontsize=13)
ax.tick_params(colors='#475569'); ax.spines[:].set_color('#1e293b')
plt.tight_layout()
plt.savefig('tsne_glove.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Classificação com Keras — Embedding Layer

Compararemos três estratégias para o embedding:

| Estratégia | `trainable` | Inicialização |
|---|---|---|
| GloVe **frozen** | `False` | pesos GloVe |
| GloVe **fine-tuned** | `True` | pesos GloVe |
| Do **zero** | `True` | aleatório |

In [ ]:
MAX_LENGTH = 300  # comprimento de sequência (cobre 90% das letras)
MAX_TOKENS = 5000

text_vec_int = keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode='int',
    output_sequence_length=MAX_LENGTH
)
text_vec_int.adapt(train_df['Lyric'])

X_train_int = text_vec_int(train_df['Lyric'])
X_val_int   = text_vec_int(val_df['Lyric'])
X_test_int  = text_vec_int(test_df['Lyric'])

print(f'Shape: {X_train_int.shape}  (letras → sequências de {MAX_LENGTH} inteiros)')
print(f'Exemplo: {X_train_int[0].numpy()[:15]} ...')
print('(0 = PAD, 1 = UNK)')

In [ ]:
# Constrói matriz de embedding GloVe alinhada ao vocabulário do TextVectorization
vocabulary = text_vec_int.get_vocabulary()
word_index = {w: i for i, w in enumerate(vocabulary)}

embedding_matrix = np.zeros((MAX_TOKENS, EMBED_DIM), dtype='float32')
found = 0
for word, i in word_index.items():
    if i < MAX_TOKENS:
        vec = embeddings_index.get(word)
        if vec is not None:
            embedding_matrix[i] = vec
            found += 1

print(f'Palavras com vetor GloVe: {found:,}/{MAX_TOKENS:,} ({100*found/MAX_TOKENS:.1f}%)')
print(f'Primeiras 3 linhas (PAD, UNK, palavra):  todas zeros = {np.all(embedding_matrix[:2] == 0)}')

In [ ]:
def build_embed_model(embedding_layer):
    inp = keras.Input(shape=(MAX_LENGTH,))
    x   = embedding_layer(inp)                       # (batch, 300, 100)
    x   = keras.layers.GlobalAveragePooling1D()(x)   # (batch, 100)
    x   = keras.layers.Dense(8, activation='relu')(x)
    x   = keras.layers.Dropout(0.5)(x)
    out = keras.layers.Dense(3, activation='softmax')(x)
    model = keras.Model(inp, out)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

def run_experiment(label, embedding_layer, epochs=10):
    print(f'\n=== {label} ===')
    model = build_embed_model(embedding_layer)
    history = model.fit(
        X_train_int, y_train_oh,
        validation_data=(X_val_int, y_val_oh),
        epochs=epochs, batch_size=32, verbose=0
    )
    _, acc = model.evaluate(X_test_int, y_test_oh, verbose=0)
    val_accs = history.history['val_accuracy']
    print(f'  Acurácia teste: {acc:.4f}  |  melhor val: {max(val_accs):.4f} (época {np.argmax(val_accs)+1})')
    return model, history, acc

In [ ]:
# ── Experimento 1: GloVe frozen ───────────────────────────────────────────────
emb_frozen = keras.layers.Embedding(
    MAX_TOKENS, EMBED_DIM,
    embeddings_initializer=keras.initializers.Constant(embedding_matrix),
    trainable=False,
    mask_zero=True
)
model_frozen, hist_frozen, acc_frozen = run_experiment('GloVe Frozen', emb_frozen)

# Confirma que os pesos NÃO mudaram
print('  Pesos PAD após treino (deve ser zero):', np.all(model_frozen.layers[1].weights[0][0].numpy() == 0))

In [ ]:
# ── Experimento 2: GloVe fine-tuned ──────────────────────────────────────────
emb_finetune = keras.layers.Embedding(
    MAX_TOKENS, EMBED_DIM,
    embeddings_initializer=keras.initializers.Constant(embedding_matrix),
    trainable=True,   # backprop atualiza os pesos!
    mask_zero=True
)
model_ft, hist_ft, acc_ft = run_experiment('GloVe Fine-tuned', emb_finetune)

print('  Pesos mudaram?', not np.allclose(
    model_ft.layers[1].weights[0][5].numpy(),
    embedding_matrix[5]
))

In [ ]:
# ── Experimento 3: Embedding do zero ─────────────────────────────────────────
emb_scratch = keras.layers.Embedding(
    MAX_TOKENS, 64,   # dimensão menor — sem conhecimento prévio
    mask_zero=True
)
model_sc, hist_sc, acc_sc = run_experiment('Embedding do Zero (64d)', emb_scratch)

In [ ]:
# ── Curvas de aprendizado comparativas ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0f172a')

configs = [
    (hist_frozen, '#38bdf8', 'GloVe frozen'),
    (hist_ft,     '#a78bfa', 'GloVe fine-tuned'),
    (hist_sc,     '#fb923c', 'Scratch 64d'),
]

for ax in axes:
    ax.set_facecolor('#1e293b'); ax.spines[:].set_color('#334155')
    ax.tick_params(colors='#94a3b8'); ax.yaxis.label.set_color('#94a3b8')
    ax.xaxis.label.set_color('#94a3b8'); ax.title.set_color('white')

epochs_x = range(1, 11)
for hist, color, label in configs:
    axes[0].plot(epochs_x, hist.history['val_accuracy'], 'o-', color=color, label=label, ms=4)

axes[0].set_title('Val Accuracy por Época'); axes[0].set_xlabel('Época'); axes[0].set_ylabel('Accuracy')
axes[0].legend(fontsize=9, facecolor='#0f172a', labelcolor='white')

# Barras
accs_all   = [acc_bow, acc_bi, acc_frozen, acc_ft, acc_sc]
labels_bar = ['BoW\nunigram', 'BoW\nbigram', 'GloVe\nfrozen', 'GloVe\nfine-tuned', 'Scratch\n64d']
bar_colors = ['#64748b', '#94a3b8', '#38bdf8', '#a78bfa', '#fb923c']

bars = axes[1].bar(labels_bar, accs_all, color=bar_colors, width=0.6)
for bar, acc in zip(bars, accs_all):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{acc:.3f}', ha='center', va='bottom', color='white', fontsize=9, fontweight='bold')
axes[1].set_title('Acurácia no Teste'); axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(min(accs_all) - 0.05, 1.0)

plt.suptitle('Letras de Música — Comparação de Abordagens', color='white', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('comparison_embeddings.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Classificação com PyTorch — `nn.Embedding`

Reimplementamos o mesmo modelo em PyTorch para comparação direta com o código dos slides.

In [ ]:
MAX_LEN_PT = 300
VOCAB_PT   = 5000

# Vocabulário PyTorch a partir dos dados de treino
def simple_tokenize(text):
    return re.sub(r'[^\w\s]', '', text.lower()).split()[:MAX_LEN_PT]

all_tokens = [simple_tokenize(t) for t in train_df['Lyric']]
counter    = Counter(w for toks in all_tokens for w in toks)
vocab_list = ['<PAD>', '<UNK>'] + [w for w, _ in counter.most_common(VOCAB_PT - 2)]
w2i        = {w: i for i, w in enumerate(vocab_list)}

def encode_pad(text, max_len=MAX_LEN_PT):
    toks = simple_tokenize(text)
    ids  = [w2i.get(t, 1) for t in toks][:max_len]
    ids += [0] * (max_len - len(ids))
    return ids

class LyricDS(Dataset):
    def __init__(self, df, labels_int):
        self.X = torch.tensor([encode_pad(t) for t in df['Lyric']], dtype=torch.long)
        self.y = torch.tensor(labels_int, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_dl_pt = DataLoader(LyricDS(train_df, y_train_int), batch_size=128, shuffle=True)
val_dl_pt   = DataLoader(LyricDS(val_df,   y_val_int),   batch_size=128)
test_dl_pt  = DataLoader(LyricDS(test_df,  y_test_int),  batch_size=128)
print(f'Vocabulário PT: {len(w2i):,} tokens')

In [ ]:
# Matriz GloVe para o vocabulário PyTorch
glove_matrix_pt = np.zeros((VOCAB_PT, EMBED_DIM), dtype='float32')
for word, idx in w2i.items():
    if idx < VOCAB_PT:
        vec = embeddings_index.get(word)
        if vec is not None:
            glove_matrix_pt[idx] = vec

class TextClassifierPT(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout = nn.Dropout(0.3)
        self.fc1     = nn.Linear(embed_dim, 64)
        self.fc2     = nn.Linear(64, num_classes)

    def forward(self, x):                               # x: (batch, seq)
        mask = (x != 0).float().unsqueeze(-1)           # ignora PAD
        e    = self.embed(x)                            # (batch, seq, d)
        e    = (e * mask).sum(1) / mask.sum(1).clamp(1) # GAP mascarado
        return self.fc2(torch.relu(self.fc1(self.dropout(e))))

def train_pt(model, epochs=10):
    opt = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    model.to(device)
    for ep in range(1, epochs + 1):
        model.train()
        for X, y in train_dl_pt:
            X, y = X.to(device), y.to(device)
            loss = crit(model(X), y)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for X, y in val_dl_pt:
                X, y = X.to(device), y.to(device)
                correct += (model(X).argmax(1) == y).sum().item(); total += len(y)
        if ep % 2 == 0 or ep == epochs:
            print(f'  Época {ep:2d}  val_acc={correct/total:.4f}')
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X, y in test_dl_pt:
            X, y = X.to(device), y.to(device)
            correct += (model(X).argmax(1) == y).sum().item(); total += len(y)
    print(f'  → Test acc: {correct/total:.4f}')
    return correct / total

print('=== PyTorch — GloVe fine-tuned ===')
model_pt = TextClassifierPT(VOCAB_PT, EMBED_DIM, num_classes=3)
model_pt.embed.weight = nn.Parameter(torch.tensor(glove_matrix_pt))
train_pt(model_pt)

---
## 9. Resumo e Próximos Passos

| Abordagem | Acurácia (aprox.) | Comentário |
|---|---|---|
| BoW unigrama + Dense | ~65% | Rápido, sem semântica |
| BoW bigrama + Dropout | ~70% | Contexto local |
| GloVe frozen + GAP | ~72% | Conhecimento geral, sem adaptação |
| GloVe fine-tuned + GAP | ~75% | Melhor com dados suficientes |
| Embedding do zero + GAP | ~73% | Surpreendentemente competitivo |

> **Regra prática** (Lec06): Se você tem *muitos* dados, embeddings pré-treinados podem não ajudar muito. Mas quando os dados são escassos, eles são muito valiosos.

In [ ]:
# ── O problema de ordem com GAP ───────────────────────────────────────────────
phrases = [
    "the dog bit the man",
    "the man bit the dog",
]

def embed_gap_glove(phrase):
    tokens = phrase.lower().split()
    vecs   = [embeddings_index[t] for t in tokens if t in embeddings_index]
    return np.mean(vecs, axis=0) if vecs else np.zeros(EMBED_DIM)

v1, v2 = embed_gap_glove(phrases[0]), embed_gap_glove(phrases[1])
sim    = cosine_sim(v1, v2)

print(f'Frase 1: "{phrases[0]}"')
print(f'Frase 2: "{phrases[1]}"')
print(f'\nCosine (GAP): {sim:.4f}  →  representações quase idênticas!')
print('\n→ GAP não distingue frases com as mesmas palavras em ordens diferentes.')
print('→ Precisamos de arquiteturas que processem a SEQUÊNCIA:')
print('   Aula 5 — RNNs e LSTMs')
print('   Aula 6 — Transformers e BERT')